# Gamma Agent

The Gamma Agent is an AI-powered content creation assistant that generates polished presentations, documents, and social media posts using the [Gamma.app](https://gamma.app) API. Describe what you need and the agent builds a fully designed, shareable Gamma link — no slide-by-slide manual work required.

In [24]:
!pip install -q aixplain requests

## Add your API keys

Get your aiXplain access key from the [Integrations](https://platform.aixplain.com/account/integrations) page.

Get your Gamma API key from [gamma.app](https://gamma.app) → Settings → API.

In [48]:
AixplainAPIKey = "d0d1ea8a13cfe80dbb34378c3941e5df13d724e1059fa065d1a802dacd4ade15"
GammaApiKey    = "sk-gamma-r8PkzyzJFUJmI4R0UdRYAzCKb11a0rhEyQqOdOBsAU"

import os
os.environ["TEAM_API_KEY"] = AixplainAPIKey
os.environ["GAMMA_API_KEY"] = GammaApiKey

In [26]:
import inspect
from aixplain import Aixplain

aix = Aixplain(AixplainAPIKey)

# What is the Gamma Agent?

The goal of this agent is to turn ideas into fully designed, shareable content in seconds. Instead of manually building slides or documents, you describe what you need and the agent handles the rest.

## Agent Architecture

The agent uses three custom script tools, one per content type:

- **Create Presentation** — generates a slide deck in 16:9, 4:3, or fluid format. Ideal for pitch decks, talks, and reports.
- **Create Document** — generates a long-form document. Ideal for reports, proposals, and briefs.
- **Create Social Post** — generates a social media carousel in 1:1, 4:5, or 9:16 format.

All three tools handle the full async flow: submit the generation request, poll for completion, and return the final shareable Gamma URL.

## Agent Workflow

1. **Understand the request** — identify content type, topic, tone, audience, and format.
2. **Call the right tool** — route to presentation, document, or social post.
3. **Return the link** — deliver the shareable Gamma URL when generation is complete.

In [39]:
def _gamma_generate(payload: dict, api_key: str) -> str:
    """Shared helper: submits a Gamma generation request and polls until complete."""
    import requests
    import time

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    try:
        resp = requests.post(
            "https://public-api.gamma.app/v1.0/generations",
            headers=headers,
            json=payload,
            timeout=30,
        )
        resp.raise_for_status()
    except requests.exceptions.HTTPError as e:
        return f"HTTP error: {e}\nResponse: {e.response.text if e.response else 'N/A'}"
    except requests.exceptions.RequestException as e:
        return f"Request error: {e}"

    generation_id = resp.json().get("id")
    if not generation_id:
        return f"Error: No generation ID returned. Response: {resp.text}"

    # Poll until complete (max 3 minutes)
    poll_url = f"https://public-api.gamma.app/v1.0/generations/{generation_id}"
    for _ in range(36):
        time.sleep(5)
        try:
            status_resp = requests.get(poll_url, headers=headers, timeout=15)
            status_resp.raise_for_status()
            data = status_resp.json()
        except requests.exceptions.RequestException as e:
            return f"Polling error: {e}"

        status = data.get("status", "")
        if status == "complete":
            url = data.get("url") or data.get("shareUrl") or data.get("viewUrl", "")
            title = data.get("title", "Untitled")
            return f"✅ {title}\n{url}"
        elif status in ("failed", "error"):
            return f"Generation failed: {data.get('error', 'Unknown error')}"

    return "Timed out waiting for Gamma to finish. Check your Gamma dashboard for the result."

In [40]:
def create_presentation(topic: str, card_count: int = 10,
                        tone: str = "professional", audience: str = "general",
                        image_source: str = "ai", aspect_ratio: str = "16x9") -> str:
    """
    Generates an AI-powered presentation using Gamma.app.

    Parameters:
        topic (str): The subject or prompt for the presentation.
                     Example: 'The future of electric vehicles', 'Q3 sales results for ACME Corp'
        card_count (int): Number of slides to generate (5–20). Default: 10.
        tone (str): Writing tone — 'professional', 'casual', 'educational', 'persuasive'.
                    Default: 'professional'.
        audience (str): Target audience — 'general', 'executives', 'technical', 'students'.
                        Default: 'general'.
        image_source (str): Image sourcing — 'ai' (AI-generated), 'web', or 'none'.
                            Default: 'ai'.
        aspect_ratio (str): Slide dimensions — '16x9' (default), '4x3', or 'fluid'.

    Returns:
        str: Title and shareable Gamma URL of the generated presentation.
    """
    import os

    api_key = os.environ.get("GAMMA_API_KEY")
    if not api_key:
        return "Error: GAMMA_API_KEY environment variable is not set."

    payload = {
        "format": "presentation",
        "prompt": topic,
        "cardCount": max(5, min(card_count, 20)),
        "tone": tone,
        "audience": audience,
        "imageSource": image_source,
        "aspectRatio": aspect_ratio,
    }
    return _gamma_generate(payload, api_key)

In [41]:
def create_document(topic: str, card_count: int = 8,
                    tone: str = "professional", audience: str = "general",
                    image_source: str = "ai") -> str:
    """
    Generates an AI-powered document using Gamma.app.

    Parameters:
        topic (str): The subject or prompt for the document.
                     Example: 'Product requirements for a mobile payment app',
                              'Quarterly business review report'
        card_count (int): Number of sections/pages to generate (3–15). Default: 8.
        tone (str): Writing tone — 'professional', 'casual', 'educational', 'persuasive'.
                    Default: 'professional'.
        audience (str): Target audience — 'general', 'executives', 'technical', 'students'.
                        Default: 'general'.
        image_source (str): Image sourcing — 'ai', 'web', or 'none'. Default: 'ai'.

    Returns:
        str: Title and shareable Gamma URL of the generated document.
    """
    import os

    api_key = os.environ.get("GAMMA_API_KEY")
    if not api_key:
        return "Error: GAMMA_API_KEY environment variable is not set."

    payload = {
        "format": "document",
        "prompt": topic,
        "cardCount": max(3, min(card_count, 15)),
        "tone": tone,
        "audience": audience,
        "imageSource": image_source,
    }
    return _gamma_generate(payload, api_key)

In [42]:
def create_social_post(topic: str, card_count: int = 6,
                       tone: str = "casual", audience: str = "general",
                       aspect_ratio: str = "1x1") -> str:
    """
    Generates an AI-powered social media carousel using Gamma.app.

    Parameters:
        topic (str): The subject or prompt for the social post.
                     Example: '5 tips for better sleep', 'How to use RAG in your AI app'
        card_count (int): Number of carousel cards to generate (3–10). Default: 6.
        tone (str): Writing tone — 'casual', 'professional', 'educational', 'persuasive'.
                    Default: 'casual'.
        audience (str): Target audience — 'general', 'executives', 'technical', 'students'.
                        Default: 'general'.
        aspect_ratio (str): Card format — '1x1' (square, default), '4x5' (portrait),
                            or '9x16' (stories/reels).

    Returns:
        str: Title and shareable Gamma URL of the generated social post.
    """
    import os

    api_key = os.environ.get("GAMMA_API_KEY")
    if not api_key:
        return "Error: GAMMA_API_KEY environment variable is not set."

    payload = {
        "format": "social",
        "prompt": topic,
        "cardCount": max(3, min(card_count, 10)),
        "tone": tone,
        "audience": audience,
        "aspectRatio": aspect_ratio,
    }
    return _gamma_generate(payload, api_key)

In [43]:

import inspect, warnings
from aixplain.v2.integration import Integration
from aixplain.v2 import integration as _integ_mod

SCRIPT_INTEGRATION_ID = "688779d8bfb8e46c273982ca"

# ── SDK workaround ────────────────────────────────────────────────────────────
# The aiXplain SDK has a dataclasses_json bug: the `attributes` field on Tool
# (and Integration) is annotated as List[Attribute] but the API returns a dict.
# When Tool.from_dict() iterates that dict it gets string keys, then tries
# `str.items()` → AttributeError.  The tool IS created on the server; we just
# can't deserialize the response.  Patch Integration.connect() to recover the
# tool id from the run response instead of re-fetching via the broken get().

class _ToolStub:
    """Minimal stand-in returned by patched connect() when Tool.get() fails."""
    redirect_url = None
    def __init__(self, id): self.id = id
    def __getattr__(self, _): return None   # all other field reads return None

_orig_connect = _integ_mod.Integration.connect

def _patched_connect(self, **kwargs):
    response = self.run(**kwargs)
    try:
        tool = self.context.Tool.get(response.data.id)
    except Exception:
        tool = _ToolStub(response.data.id)
    if getattr(response.data, "redirectURL", None):
        tool.redirect_url = response.data.redirectURL
        warnings.warn(f"Visit this URL to complete authentication: {response.data.redirectURL}")
    return tool

_integ_mod.Integration.connect = _patched_connect
# ─────────────────────────────────────────────────────────────────────────────

# Build an Integration object directly — avoids the same bug in Integration.get()
script_integration = Integration(id=SCRIPT_INTEGRATION_ID, name="Python Sandbox")
script_integration.context = aix.Integration.context

helper_src = inspect.getsource(_gamma_generate)

def _inject(fn):
    src = helper_src + "\n\n" + inspect.getsource(fn)
    return src.replace('os.environ.get("GAMMA_API_KEY")', f'"{GammaApiKey}"')

presentation_tool = aix.Tool(
    name="Create Gamma Presentation 2",
    integration=script_integration,
    config={"code": _inject(create_presentation), "function_name": "create_presentation"},
)
presentation_tool.save()
print(f"Created: {presentation_tool.name}  id={presentation_tool.id}")

document_tool = aix.Tool(
    name="Create Gamma Document 2",
    integration=script_integration,
    config={"code": _inject(create_document), "function_name": "create_document"},
)
document_tool.save()
print(f"Created: {document_tool.name}  id={document_tool.id}")

social_tool = aix.Tool(
    name="Create Gamma Social Post 2",
    integration=script_integration,
    config={"code": _inject(create_social_post), "function_name": "create_social_post"},
)
social_tool.save()
print(f"Created: {social_tool.name}  id={social_tool.id}")


Created: Create Gamma Presentation 2  id=69d5646089421107e687b82b
Created: Create Gamma Document 2  id=69d564611f5533df7784daba
Created: Create Gamma Social Post 2  id=69d564626873711daf566fc6


# Build Your Agent

To create an agent, define:

* A unique **name** and **description** for its purpose.
* The **tools** it will use — presentation, document, and social post generators.
* The **instructions** that guide the agent's content type selection and parameter choices.

In [45]:
gamma_agent = aix.Agent(
    name="Gamma Agent 1",
    description="AI content creation agent that generates polished presentations, documents, and social media carousels using Gamma.app. Returns a shareable link for each piece of content.",
    instructions="""
    You are a content creation assistant with access to Gamma.app's generation API.
    You have three tools — choose based on what the user wants to create:

    - Create Gamma Presentation: for slide decks, pitch decks, talks, and presentations.
      Default aspect ratio 16x9. Use 'persuasive' tone for pitch decks.

    - Create Gamma Document: for reports, proposals, briefs, and long-form content.
      No aspect ratio needed. Good for structured written content.

    - Create Gamma Social Post: for Instagram carousels, LinkedIn posts, and social content.
      Use 1x1 for LinkedIn/Instagram square, 9x16 for stories/reels.

    Parameter guidelines:
    - tone: 'professional' for business content, 'casual' for social, 'educational' for tutorials
    - audience: extract from the request — 'executives', 'technical', 'students', or 'general'
    - card_count: infer from complexity — short topics 5–7 cards, detailed topics 10–15
    - image_source: default to 'ai' unless the user says no images

    Always return the full Gamma URL as plain text so it is clickable.
    Generation typically takes 20–60 seconds.
    """,
    tools=[presentation_tool, document_tool, social_tool],
)
gamma_agent.save()
print(f"Agent created: {gamma_agent.id}")

Agent created: 69d5647478871c963dd79201


# Run your Agent

To ensure your agent is performing as expected, run it using the `run` method with sample inputs. Analyze the outputs, verify their accuracy, and debug any issues by inspecting the agent's configurations, tools, and instructions.

In [49]:

# Sanity-check the Gamma API before running the agent
import requests

_headers = {
    "Authorization": f"Bearer {GammaApiKey}",
    "Content-Type": "application/json",
}
_payload = {
    "format": "presentation",
    "prompt": "Test",
    "cardCount": 5,
    "tone": "professional",
    "audience": "general",
    "imageSource": "ai",
    "aspectRatio": "16x9",
}
_resp = requests.post(
    "https://public-api.gamma.app/v1.0/generations",
    headers=_headers,
    json=_payload,
    timeout=30,
)
print(f"Status: {_resp.status_code}")
print(f"Response: {_resp.text[:500]}")


Status: 401
Response: {"message":{"error":"invalid_token","error_description":"The access token is invalid or expired","statusCode":401},"statusCode":401}


In [50]:

# Pitch deck
result = None
try:
    result = gamma_agent.run(
        "Create a 10-slide pitch deck for an AI-powered recipe app targeting investors"
    )
    print(result.data.output)
except Exception as e:
    print(f"Agent run failed: {e}")


Agent run failed: Operation failed: {'code': 'AX-SUP-1000', 'description': 'The model response was cut off because the maximum token limit was reached. Please increase your `max_tokens` setting and try again.', 'category': 'supplier', 'failed_tool': None, 'tool_input': None}


In [ ]:

# Document
result2 = None
try:
    result2 = gamma_agent.run(
        "Generate a product requirements document for a mobile payment feature"
    )
    print(result2.data.output)
except Exception as e:
    print(f"Agent run failed: {e}")


In [ ]:

# Social carousel
result3 = None
try:
    result3 = gamma_agent.run(
        "Make a LinkedIn carousel: 5 tips for building better AI agents"
    )
    print(result3.data.output)
except Exception as e:
    print(f"Agent run failed: {e}")


In [ ]:

# Multi-turn: create a matching social post for the pitch deck
if result is None:
    print("Skipping multi-turn example: initial pitch deck run failed.")
else:
    result4 = None
    try:
        result4 = gamma_agent.run(
            "Now make a short Instagram carousel announcing the same app",
            session_id=result.data.session_id,
        )
        print(result4.data.output)
    except Exception as e:
        print(f"Agent run failed: {e}")


# Save the Agent

Once you are happy with your agent, save it to access the agent endpoints.

In [ ]:
gamma_agent.save()

aiXplain empowers you to seamlessly build, customize, and deploy intelligent agents tailored to your specific needs. Whether you're creating standalone agents or advanced multi-agent systems, the platform provides a robust toolkit for integrating cutting-edge AI capabilities into your workflows. To learn more, visit [aiXplain](https://aixplain.com).